# 00 — Fetch Data OpenF1

Semua kode ada di notebook ini. Cukup atur tiga nilai di cell **Jalankan fetch**, lalu **Run All**.

Hasil fetch disimpan ke `data/raw/` dan dipakai oleh `01_tables.ipynb`.

## Pengenalan Data — Endpoint yang Dipakai

Proyek ini memprediksi **posisi grid kualifikasi**. Karena itu yang diambil hanya data yang tersedia **sebelum Qualifying dimulai** (fitur) plus **hasil Qualifying** (target).

| Endpoint | Apa isinya | Kenapa dipakai |
|---|---|---|
| `sessions` | Daftar sesi (FP1, FP2, FP3, Qualifying, Race) lengkap dengan `session_key`, tanggal, sirkuit | **Pintu masuk.** Tanpa ini tidak tahu `session_key` sesi mana yang perlu diambil data lanjutannya. Juga jadi sumber metadata `year` dan `circuit_short_name`. |
| `laps` | Satu baris per lap: waktu lap, sektor, kecepatan (`i1_speed`, `i2_speed`, `st_speed`) | **Fitur utama.** Dari lap FP1–FP3 dihitung pace pembalap sebelum kualifikasi. Lap Qualifying juga disimpan untuk analisis pacing kualifikasi. |
| `stints` | Periode pemakaian ban: `compound` (SOFT/MEDIUM/HARD), `lap_start`, `lap_end`, umur ban | **Konteks pace FP.** Lap FP dengan ban baru lebih cepat dari ban tua; tanpa info ini, perbandingan pace antar pembalap jadi menyesatkan. |
| `weather` | Kondisi cuaca per sesi: suhu udara/track, kelembapan, angin, hujan | **Fitur.** Cuaca memengaruhi pace — pembalap cepat di FP basah belum tentu cepat di quali kering, dan sebaliknya. |
| `drivers` | Identitas pembalap: nomor, nama, `team_name` | **Fitur kategori + identitas.** Tim menentukan performa mobil; dipakai untuk menghubungkan lap ke pembalap/tim. |
| `starting_grid` | Hasil kualifikasi: posisi grid + waktu lap tiap pembalap | **TARGET.** Ini yang ingin diprediksi model: `position` = posisi grid. Diambil dari `session_key` sesi Qualifying (bukan Race). |

### Endpoint yang TIDAK dipakai (dan alasannya)

| Endpoint | Alasan tidak diambil |
|---|---|
| `session_result` | Hasil akhir **race** — terjadi setelah qualifying, bukan fitur |
| `pit` | Pit stop **race** — setelah qualifying |
| `race_control` | Insiden/safety car **race** — setelah qualifying |
| `championship_drivers`, `championship_teams` | Klasemen yang dihitung dari hasil race — mengandung info setelah qualifying (leakage) |
| `car_data`, `location` | Telemetry mentah, ~jutaan baris per race — terlalu berat, tidak perlu untuk prediksi grid |
| `team_radio`, `overtakes` | Tidak relevan untuk prediksi pre-quali; cakupan data juga tidak lengkap |

**Aturan utama:** semua fitur harus berasal dari titik waktu **sebelum Qualifying**. Data setelah Qualifying (hasil race, pit, klasemen) tidak boleh masuk sebagai fitur — itu namanya *leakage*.

## 0. Setup & konfigurasi

Cell ini:

- mencari root project otomatis (aman kalau notebook dibuka dari folder `notebooks/`),
- membaca `config.yaml` menjadi variabel `cfg`,
- membuat `resolve_path()` supaya path relatif selalu dihitung dari root project.

Output yang diharapkan: satu baris `Project root: ...`.

In [1]:
import logging
import time
from pathlib import Path

import pandas as pd
import requests
import yaml

# Cari root project. Aman dijalankan dari folder notebooks/ maupun dari root.
ROOT = Path.cwd()
if not (ROOT / "config.yaml").exists() and (ROOT.parent / "config.yaml").exists():
    ROOT = ROOT.parent

with (ROOT / "config.yaml").open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle) or {}


def resolve_path(value):
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


print("Project root:", ROOT)

Project root: c:\syntax\codelab\SubrisetData\prediksi-openf1


## 1. Cache `data/raw/`

Fungsi di cell ini menyimpan tiap respons API ke:

```text
data/raw/<endpoint>/<key>.parquet
```

- `load_cached()` dicek lebih dulu. Kalau file sudah ada, API tidak dipanggil ulang.
- `save_cached()` memakai Parquet. Kalau ada kolom bertipe campuran (misal `gap_to_leader` berisi angka dan `"+1 LAP"`), file-nya fallback ke `.json`.
- `cached_get(..., force=True)` bisa dipakai untuk memaksa fetch ulang.

Tidak ada output di cell ini.

In [2]:
# ---- Cache: simpan respons API ke data/raw/ ----

def cache_path(raw_dir, endpoint, key, suffix=".parquet"):
    return Path(raw_dir) / endpoint / f"{key}{suffix}"


def to_dataframe(payload):
    if payload is None:
        return pd.DataFrame()
    if isinstance(payload, list):
        return pd.DataFrame(payload)
    if isinstance(payload, dict):
        return pd.DataFrame([payload])
    return pd.DataFrame(payload)


def load_cached(raw_dir, endpoint, key):
    parquet_path = cache_path(raw_dir, endpoint, key)
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    json_path = cache_path(raw_dir, endpoint, key, ".json")
    if json_path.exists():
        return pd.read_json(json_path)
    return None


def save_cached(df, raw_dir, endpoint, key):
    path = cache_path(raw_dir, endpoint, key)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        df.to_parquet(path, index=False)
        return path
    except Exception:
        # Kolom bertipe campuran (mis. gap_to_leader = 2.664 / "+1 LAP")
        # tidak bisa ditulis ke parquet, jadi fallback ke JSON.
        fallback = path.with_suffix(".json")
        df.to_json(fallback, orient="records", date_format="iso")
        return fallback


def cached_get(client, endpoint, key, raw_dir, *, force=False, **params):
    if not force:
        cached = load_cached(raw_dir, endpoint, key)
        if cached is not None:
            return cached
    payload = client.get(endpoint, **params)
    df = to_dataframe(payload)
    save_cached(df, raw_dir, endpoint, key)
    return df

## 2. Client OpenF1

Bagian ini mengatur cara request ke API:

- rate limit 30 request/menit (diatur di `config.yaml`),
- retry dengan exponential backoff untuk error sementara,
- `401/403` → pesan jelas, biasanya karena ada sesi live,
- `404` → dianggap "data kosong", bukan error fatal,
- `429` → tunggu sebentar lalu retry.

Tidak ada output di cell ini.

In [3]:
# ---- Client OpenF1 ----
LOGGER = logging.getLogger("openf1")
PRACTICE_ENDPOINTS = ("laps", "stints", "weather", "drivers")
RACE_ENDPOINTS = (
    "laps", "stints", "weather", "drivers",
    "starting_grid", "session_result", "pit",
    "championship_drivers", "championship_teams", "race_control",
)


class OpenF1AccessError(RuntimeError):
    pass


class OpenF1Client:
    def __init__(self, base_url, timeout, max_retries, rate_limit_per_minute):
        self.base_url = base_url.rstrip("/")
        self.timeout = timeout
        self.max_retries = max_retries
        self.min_interval = 60.0 / max(rate_limit_per_minute, 1)
        self._last_request_at = 0.0
        self.session = requests.Session()
        self.session.headers.update({"User-Agent": "prediksi-openf1/0.2"})

    def _throttle(self):
        wait = self.min_interval - (time.monotonic() - self._last_request_at)
        if wait > 0:
            time.sleep(wait)
        self._last_request_at = time.monotonic()

    def get(self, endpoint, **params):
        url = f"{self.base_url}/{endpoint.lstrip('/')}"
        last_error = None
        for attempt in range(1, self.max_retries + 1):
            self._throttle()
            try:
                response = self.session.get(url, params=params, timeout=self.timeout)

                if response.status_code in (401, 403):
                    try:
                        detail = response.json().get("detail", response.text)
                    except Exception:
                        detail = response.text
                    raise OpenF1AccessError(
                        f"OpenF1 menolak request ({response.status_code}). "
                        "Biasanya ada sesi live; akses historis dibatasi sampai selesai. "
                        f"Detail: {detail}"
                    )

                # 404 = tidak ada data untuk filter ini, bukan error fatal.
                if response.status_code == 404:
                    LOGGER.warning("Tidak ada data untuk %s params=%s", endpoint, params)
                    return []

                if response.status_code == 429:
                    wait = min(60, 2**attempt)
                    LOGGER.warning("Rate limit tercapai, tunggu %ss", wait)
                    time.sleep(wait)
                    continue

                response.raise_for_status()
                payload = response.json()
                if isinstance(payload, dict) and "detail" in payload:
                    raise RuntimeError(f"API error: {payload['detail']}")
                return payload

            except OpenF1AccessError:
                raise
            except Exception as exc:
                last_error = exc
                if attempt == self.max_retries:
                    break
                wait = 2**attempt
                LOGGER.warning("Request gagal (%s), retry dalam %ss", exc, wait)
                time.sleep(wait)

        raise RuntimeError(f"Gagal mengambil {url} dengan params={params}") from last_error

    def get_sessions(self, years, circuit_key=None):
        sessions = []
        for year in years:
            payload = self.get("sessions", year=year)
            if isinstance(payload, list):
                sessions.extend(payload)
        if circuit_key is not None:
            sessions = [s for s in sessions if s.get("circuit_key") == circuit_key]
        return sessions

## 3. Fungsi `fetch()`

`fetch()` melakukan urutan ini:

1. ambil daftar sesi untuk tahun dan sirkuit yang dipilih,
2. filter nama sesi dari `config.yaml` (sprint tidak termasuk),
3. kelompokkan sesi per `meeting_key` → satu weekend,
4. simpan metadata weekend ke `data/raw/sessions/`,
5. ambil endpoint yang dibutuhkan untuk tiap sesi.

**Catatan penting:** `starting_grid` diambil dari sesi **Qualifying**, bukan Race.
Memakai `session_key` Race akan menghasilkan `404`.

In [4]:
# ---- Fungsi fetch utama ----

def select_weekend_sessions(sessions, session_names, include_sprint=False):
    allowed = set(session_names)
    if include_sprint:
        allowed.update({"Sprint", "Sprint Qualifying"})
    return [
        s for s in sessions
        if s.get("session_name") in allowed and not s.get("is_cancelled", False)
    ]


def group_by_meeting(sessions):
    grouped = {}
    for item in sessions:
        grouped.setdefault(item["meeting_key"], []).append(item)
    return {
        key: sorted(items, key=lambda item: item.get("date_start", ""))
        for key, items in grouped.items()
    }


def fetch_weekend(client, sessions, raw_dir, *, force=False):
    records = []
    qualifying = next(
        (s for s in sessions if s.get("session_name") == "Qualifying"), None
    )
    for session in sessions:
        session_key = session["session_key"]
        session_name = session.get("session_name", "")
        endpoints = RACE_ENDPOINTS if session_name == "Race" else PRACTICE_ENDPOINTS
        LOGGER.info("Fetch %s | session_key=%s", session_name, session_key)

        for endpoint in endpoints:
            if endpoint == "starting_grid":
                # OpenF1 menyimpan starting grid di sesi Qualifying, bukan Race.
                # Memakai session_key Race akan menghasilkan 404.
                if qualifying is None:
                    LOGGER.warning("Qualifying tidak ditemukan, skip starting_grid.")
                    continue
                key_session = qualifying["session_key"]
            else:
                key_session = session_key

            df = cached_get(
                client,
                endpoint,
                f"session_{key_session}",
                raw_dir,
                force=force,
                session_key=key_session,
            )
            records.append({
                "session_key": key_session,
                "session_name": "Qualifying" if endpoint == "starting_grid" else session_name,
                "endpoint": endpoint,
                "rows": len(df),
            })
    return records


def fetch(year=None, circuit_key="config", limit=None, *, force=False, include_sprint=None):
    if not logging.getLogger().handlers:
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s | %(levelname)s | %(message)s",
            datefmt="%H:%M:%S",
        )

    data_cfg = cfg.get("data", {})
    if year is None:
        years = list(data_cfg.get("years", []))
    elif isinstance(year, int):
        years = [year]
    else:
        years = list(year)
    if circuit_key == "config":
        circuit_key = data_cfg.get("circuit_key")
    if include_sprint is None:
        include_sprint = bool(data_cfg.get("include_sprint", False))

    api_cfg = cfg.get("api", {})
    client = OpenF1Client(
        base_url=api_cfg.get("base_url", "https://api.openf1.org/v1"),
        timeout=api_cfg.get("timeout_seconds", 30),
        max_retries=api_cfg.get("max_retries", 3),
        rate_limit_per_minute=api_cfg.get("rate_limit_per_minute", 30),
    )
    raw_dir = resolve_path(cfg["paths"]["raw"])
    sessions = client.get_sessions(years=years, circuit_key=circuit_key)
    selected = select_weekend_sessions(
        sessions, data_cfg.get("session_names", []), include_sprint=include_sprint
    )
    weekends = group_by_meeting(selected)

    save_cached(pd.DataFrame(selected), raw_dir, "sessions", f"circuit_{circuit_key}")

    items = list(weekends.items())
    if limit:
        items = items[:limit]
    LOGGER.info(
        "Ditemukan %s weekend (%s sesi). Memproses %s weekend.",
        len(weekends), len(selected), len(items),
    )

    records = []
    for index, (meeting_key, weekend_sessions) in enumerate(items, start=1):
        LOGGER.info("[%s/%s] Meeting %s", index, len(items), meeting_key)
        records.extend(fetch_weekend(client, weekend_sessions, raw_dir, force=force))

    LOGGER.info("Selesai. Cache tersimpan di %s", raw_dir)
    return pd.DataFrame(records)

## 4. Jalankan fetch

Ubah tiga nilai di bawah sesuai kebutuhan, lalu jalankan cell ini.

| Parameter | Arti |
|---|---|
| `year` | `2024`, `[2023, 2024]`, atau `None` = semua tahun di `config.yaml` |
| `circuit_key` | `39` = Monza, `None` = semua sirkuit |
| `limit` | `1` untuk tes, `None` untuk semua weekend |

Output di cell ini:

- log proses: weekend yang diproses, `meeting_key`, dan `session_key`,
- tabel `summary`: satu baris per endpoint per sesi, dengan kolom `rows` = jumlah baris respons.

In [5]:
summary = fetch(
    year=2024,       # 2024, [2023, 2024], atau None untuk semua tahun di config
    circuit_key=39,  # 39 = Monza; None = semua sirkuit
    limit=1,         # 1 weekend untuk tes; None untuk semua weekend
)

summary

OpenF1AccessError: OpenF1 menolak request (401). Biasanya ada sesi live; akses historis dibatasi sampai selesai. Detail: Live F1 session in progress. Global API access (including past sessions) is restricted to authenticated users until the session ends. Get an API key here: https://buy.stripe.com/eVqcN41BPekP0iIalBcEw02

## 5. Fetch semua weekend

Setelah tes dengan `limit=1` berhasil, ganti menjadi `limit=None`:

```python
fetch(year=None, circuit_key=39, limit=None)
```

Kalau ingin semua sirkuit:

```python
fetch(year=None, circuit_key=None, limit=None)
```

Fetch penuh memakan waktu lebih lama karena rate limit 30 request/menit.

In [ ]:
# Contoh fetch penuh Monza:
# fetch(year=None, circuit_key=39, limit=None)

## 6. Cek cache

Cell ini menghitung jumlah file per endpoint di `data/raw/`.

Contoh struktur untuk satu weekend:

```text
data/raw/
├── sessions/circuit_39.parquet
├── laps/session_9590.parquet
├── stints/session_9590.parquet
├── weather/session_9590.parquet
├── drivers/session_9590.parquet
├── starting_grid/session_9586.parquet    ← dari sesi Qualifying
├── session_result/session_9590.parquet
├── pit/session_9590.parquet
├── championship_drivers/session_9590.parquet
├── championship_teams/session_9590.parquet
└── race_control/session_9590.parquet
```

Sebagian file bisa berakhir `.json` kalau kolomnya bertipe campuran.

In [ ]:
raw = resolve_path(cfg["paths"]["raw"])
endpoints = [
    "sessions", "laps", "stints", "starting_grid", "session_result", "weather",
    "pit", "drivers", "race_control",
]
for endpoint in endpoints:
    folder = raw / endpoint
    count = (
        len(list(folder.glob("*.parquet"))) + len(list(folder.glob("*.json")))
        if folder.exists() else 0
    )
    print(f"{endpoint:20s} {count} file")

sessions             1 file
laps                 5 file
stints               5 file
starting_grid        1 file
session_result       1 file
weather              5 file
pit                  1 file
drivers              5 file
race_control         1 file


## Ringkasan output `00_fetch.ipynb`

| Output | Bentuk | Keterangan |
|---|---|---|
| Log proses | teks di console | weekend yang diproses + `session_key` |
| `summary` | DataFrame | 1 baris per endpoint per sesi; kolom `rows` = jumlah baris respons |
| `data/raw/**` | file Parquet/JSON | cache mentah, dipakai oleh `01_tables.ipynb` |

Tabel olahan belum dibuat di notebook ini. Lanjut ke `01_tables.ipynb`.